<a href="https://colab.research.google.com/github/RMoulla/Python_AFD/blob/main/RAG_Demo_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG PDF depuis Google Drive vers FAISS + interface de chat

Ce notebook Colab :

- monte Google Drive
- lit tous les PDF du dossier `MyDrive/Demo`
- découpe les documents en chunks
- calcule les embeddings
- sauvegarde l'index vectoriel dans FAISS
- lance une interface de chat Gradio propre et agréable

> **Remplacer simplement la clé API OpenAI dans la cellule de configuration entre les deux "".**

In [1]:
!pip -q install -U langchain langchain-community langchain-openai faiss-cpu pypdf
!pip -q install -U langchain-core
!pip install -q gradio==5.50.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 513.0/513.0 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [2]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
import os
from pathlib import Path

# ========= Configuration =========

OPENAI_API_KEY = ""


# Dossier PDF dans Google Drive
PDF_DIR = Path("/content/drive/MyDrive/Demo")

# Dossier de persistance FAISS
INDEX_DIR = Path("/content/drive/MyDrive/Demo_faiss_index")

# Paramètres de chunking
CHUNK_SIZE = 1200
CHUNK_OVERLAP = 200

# Modèles
EMBEDDING_MODEL = "text-embedding-3-small"
CHAT_MODEL = "gpt-4o-mini"

# Retrieval
TOP_K = 4

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

assert PDF_DIR.exists(), f"Le dossier {PDF_DIR} n'existe pas."
assert OPENAI_API_KEY and OPENAI_API_KEY != "PASTE_YOUR_OPENAI_API_KEY_HERE", "Ajoute ta clé API OpenAI."
print("Configuration OK")

Configuration OK


In [4]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

def load_pdf_documents(pdf_dir: Path):
    pdf_files = sorted(pdf_dir.rglob("*.pdf"))
    if not pdf_files:
        raise ValueError(f"Aucun PDF trouvé dans : {pdf_dir}")

    docs = []
    for pdf_path in pdf_files:
        loader = PyPDFLoader(str(pdf_path))
        pages = loader.load()
        for page in pages:
            page.metadata["source_file"] = pdf_path.name
            page.metadata["source_path"] = str(pdf_path)
        docs.extend(pages)

    print(f"{len(pdf_files)} PDF(s) trouvé(s)")
    print(f"{len(docs)} page(s) chargée(s)")
    return docs

def chunk_documents(documents):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=["\n\n", "\n", ". ", " ", ""]
    )
    chunks = splitter.split_documents(documents)
    print(f"{len(chunks)} chunk(s) généré(s)")
    return chunks

In [5]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

def build_or_load_vectorstore(force_rebuild: bool = False):
    embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

    if INDEX_DIR.exists() and not force_rebuild:
        print(f"Chargement de l'index existant depuis {INDEX_DIR}")
        # À utiliser uniquement si tu fais confiance à l'index local
        vectorstore = FAISS.load_local(
            folder_path=str(INDEX_DIR),
            embeddings=embeddings,
            allow_dangerous_deserialization=True
        )
        return vectorstore

    print("Construction d'un nouvel index...")
    docs = load_pdf_documents(PDF_DIR)
    chunks = chunk_documents(docs)

    vectorstore = FAISS.from_documents(chunks, embeddings)
    INDEX_DIR.mkdir(parents=True, exist_ok=True)
    vectorstore.save_local(str(INDEX_DIR))
    print(f"Index FAISS sauvegardé dans : {INDEX_DIR}")
    return vectorstore

vectorstore = build_or_load_vectorstore(force_rebuild=False)
retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})

Chargement de l'index existant depuis /content/drive/MyDrive/Demo_faiss_index


In [6]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(model=CHAT_MODEL, temperature=0)

SYSTEM_PROMPT = '''
Tu es un assistant RAG précis, clair et honnête.
Réponds uniquement à partir du contexte fourni.
Si l'information n'est pas dans le contexte, dis-le explicitement.
Quand c'est utile, structure la réponse de manière lisible.
Réponds en français sauf si l'utilisateur demande une autre langue.
'''

PROMPT = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", '''
Question utilisateur :
{question}

Contexte :
{context}

Réponse :
''')
])

def format_context(docs):
    blocks = []
    for i, doc in enumerate(docs, start=1):
        source = doc.metadata.get("source_file", "source inconnu")
        page = doc.metadata.get("page", "?")
        content = doc.page_content.strip()
        blocks.append(f"[Source {i} | fichier={source} | page={page}]\n{content}")
    return "\n\n".join(blocks)

def answer_question(question: str):
    retrieved_docs = retriever.invoke(question)
    context = format_context(retrieved_docs)
    messages = PROMPT.format_messages(question=question, context=context)
    response = llm.invoke(messages)
    return response.content, retrieved_docs

In [7]:
import html

def render_sources_markdown(docs):
    if not docs:
        return "Aucune source retrouvée."

    blocks = []
    for i, doc in enumerate(docs, start=1):
        source = doc.metadata.get("source_file", "source inconnu")
        path = doc.metadata.get("source_path", "")
        page = doc.metadata.get("page", "?")
        excerpt = doc.page_content[:900].strip().replace("\n", " ")
        blocks.append(
            f"### Source {i}\n"
            f"- **Fichier** : `{source}`\n"
            f"- **Page** : `{page}`\n"
            f"- **Chemin** : `{path}`\n"
            f"- **Extrait** : {html.escape(excerpt)}...\n"
        )
    return "\n\n".join(blocks)

In [ ]:
import gradio as gr

CUSTOM_CSS = '''
body, .gradio-container {
    background:
        radial-gradient(circle at top left, rgba(59,130,246,0.16), transparent 28%),
        radial-gradient(circle at top right, rgba(168,85,247,0.14), transparent 30%),
        linear-gradient(135deg, #020617 0%, #0f172a 45%, #111827 100%) !important;
    color: #e5e7eb !important;
}

/* container */
.gradio-container {
    max-width: 1180px !important;
    margin: auto !important;
}

/* titre */
#app-title {
    text-align: center;
    font-size: 34px;
    font-weight: 900;
    margin-bottom: 6px;
    background: linear-gradient(90deg, #f8fafc, #93c5fd, #c4b5fd);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
}

#app-subtitle {
    text-align: center;
    color: #cbd5e1;
    margin-bottom: 18px;
}

/* panels */
.panel {
    border-radius: 22px !important;
    backdrop-filter: blur(10px);
    background: rgba(15, 23, 42, 0.75) !important;
    border: 1px solid rgba(148, 163, 184, 0.18) !important;
    box-shadow: 0 12px 30px rgba(0,0,0,0.3);
    opacity: 1 !important;
}

/* ===== FIX MARKDOWN PROPRE ===== */

/* CHAT uniquement */
.gradio-container .message .prose,
.gradio-container .message .prose * {
    color: black !important;
    background: transparent !important;
}

/* code dans chat */
.gradio-container .message pre,
.gradio-container .message code {
    background: #f3f4f6 !important;
    color: black !important;
}

/* PANELS dark */
.panel .prose,
.panel .prose * {
    color: #e5e7eb !important;
    background: transparent !important;
}

/* bulles chat */
.gradio-container .message.user {
    background: linear-gradient(180deg, #e0f2fe, #bae6fd) !important;
    color: black !important;
}

.gradio-container .message.bot {
    background: white !important;
    color: black !important;
}

/* textbox */
textarea {
    border-radius: 14px !important;
    background: white !important;
    color: black !important;
}

/* boutons */
button {
    border-radius: 14px !important;
    font-weight: 700 !important;
    background: linear-gradient(90deg, #2563eb, #7c3aed) !important;
    color: white !important;
}

/* enlever footer */
footer {
    display: none !important;
}
'''

def chatbot_fn(message, history):
    answer, docs = answer_question(message)
    sources_md = render_sources_markdown(docs)
    return answer, sources_md

def rebuild_index():
    global vectorstore, retriever
    vectorstore = build_or_load_vectorstore(force_rebuild=True)
    retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})
    return "✅ Index reconstruit avec succès."

with gr.Blocks(css=CUSTOM_CSS, theme=gr.themes.Base()) as demo:
    gr.HTML("<div id='app-title'>Application RAG • Google Drive </div>")
    gr.HTML("<div id='app-subtitle'>Interroge les PDF du dossier <b>Demo</b> dans Google Drive.</div>")

    with gr.Row():
        with gr.Column(scale=7):
            chatbot = gr.ChatInterface(
                fn=chatbot_fn,
                type="messages",
                additional_outputs=[
                    gr.Markdown(label="Sources utilisées", elem_classes=["panel"])
                ],
                fill_height=True
            )

        with gr.Column(scale=3):
            with gr.Group(elem_classes=["panel"]):
                gr.Markdown("### Paramètres")
                gr.Markdown(
                    f'''
- **Dossier PDF** : `{PDF_DIR}`
- **Index FAISS** : `{INDEX_DIR}`
- **Embedding** : `{EMBEDDING_MODEL}`
- **Chat model** : `{CHAT_MODEL}`
- **Top-k** : `{TOP_K}`
                    '''
                )

                rebuild_btn = gr.Button("Reconstruire l'index")
                rebuild_status = gr.Markdown()
                rebuild_btn.click(fn=rebuild_index, outputs=rebuild_status)

            with gr.Group(elem_classes=["panel"]):
                gr.Markdown("### Conseils")
                gr.Markdown(
                    '''
- Mets tes PDF dans `MyDrive/Demo`
- Lance *Reconstruire l'index* après ajout de nouveaux fichiers
- Pose des questions précises pour de meilleurs résultats
                    '''
                )

demo.launch(
    debug=True,
    share=True
)

/tmp/ipykernel_567/1166501246.py:111: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=CUSTOM_CSS, theme=gr.themes.Base()) as demo:
/tmp/ipykernel_567/1166501246.py:111: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=CUSTOM_CSS, theme=gr.themes.Base()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://30c4c2ed8a286c250b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
